In [1]:

from pathlib import Path

import pandas as pd
import zarr

ROOT = Path.home() / "Code/UG_SM/Data/Global"

PARQUETS = [
    ROOT / "global_ismn_daily.parquet",
    ROOT / "station_date_candidates.parquet",
    ROOT / "stations.parquet",
    ROOT / "era5_backend/era5_by_sample.parquet",
    ROOT / "era5_backend/era5_point_index.parquet",
    ROOT / "era5_backend/era5_sample_features.parquet",
    ROOT / "smap/smap_by_sample.parquet",
    ROOT / "smap/station_smap_grid_mapping.parquet",
]

CSVS = [
    ROOT / "Statics/global_static_soil.csv",
    ROOT / "Statics/global_static_terrain.csv",
]

def inspect_parquet(path):
    print(f"\n{'=' * 80}\nPARQUET: {path.relative_to(ROOT)}")
    if not path.exists():
        print("NOT FOUND")
        return

    df = pd.read_parquet(path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))

for path in PARQUETS:
    inspect_parquet(path)

for path in CSVS:
    print(f"\n{'=' * 80}\nCSV: {path.relative_to(ROOT)}")
    if not path.exists():
        print("NOT FOUND")
        continue

    df = pd.read_csv(path, nrows=3)
    print("Columns:", df.columns.tolist())
    print(df.to_string(index=False))

def inspect_zarr(path):
    print(f"\n{'=' * 80}\nZARR: {path}")
    if not path.exists():
        print("NOT FOUND")
        return

    root = zarr.open_group(str(path), mode="r")
    print("Attributes:", dict(root.attrs))

    def visit(name, obj):
        if isinstance(obj, zarr.Array):
            print(
                f"  {name}: shape={obj.shape}, "
                f"dtype={obj.dtype}, chunks={obj.chunks}"
            )

    root.visititems(visit)

for folder in [
    ROOT / "modis/evi_lst_patch_dataset",
    ROOT / "modis/modis_patch_dataset",
    ROOT / "modis/ndvi_lst_patch_dataset",
    ROOT / "sentinel1/nearest_valid_overview",
]:
    if folder.exists():
        for path in sorted(folder.iterdir()):
            if path.is_dir() and (
                path.name.startswith("20") or "zarr" in path.name.lower()
            ):
                inspect_zarr(path)



PARQUET: global_ismn_daily.parquet
Shape: (5045849, 19)
Columns: ['station_id', 'network', 'network_name', 'station', 'latitude', 'longitude', 'elevation_m', 'sensor', 'depth_from_m', 'depth_to_m', 'depth_class', 'sample_date', 'soil_moisture_m3_m3', 'observation_count', 'source_file', 'clay_pct', 'sand_pct', 'silt_pct', 'climate_zone']
             station_id    network network_name     station  latitude  longitude  elevation_m         sensor  depth_from_m  depth_to_m  depth_class sample_date  soil_moisture_m3_m3  observation_count                                                                                                            source_file  clay_pct  sand_pct  silt_pct climate_zone
AMMA-CATCH__Banizoumbou AMMA-CATCH   AMMA-CATCH Banizoumbou   13.5325     2.6604        208.0 Campbell-CS616          0.05        0.05 near_surface  2015-04-01             0.005817                 24 AMMA-CATCH/Banizoumbou/AMMA-CATCH_AMMA-CATCH_Banizoumbou_sm_0.050000_0.050000_Campbell-CS616_1_1_2

ArrowInvalid: Could not open Parquet input source '<Buffer>': Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.